# marker-pdf 결과물 테스트
PDF → Markdown 변환, 이미지 추출, 섹션 분리까지 전체 과정을 직접 확인합니다.

In [1]:
from pathlib import Path

# asset 폴더의 PDF 파일 목록
pdf_files = sorted(Path("asset").glob("*.pdf"))
for f in pdf_files:
    print(f"  - {f.name}")

# 테스트할 PDF 선택 (첫 번째 파일)
pdf_path = str(pdf_files[0])
print(f"\n선택된 PDF: {pdf_path}")

  - MSCRS_ Multi-modal Semantic Graph Prompt Learning Framework for Conversational Recommender Systems.pdf
  - MT3-MultiTask Multitrack Music transcription.pdf

선택된 PDF: asset/MSCRS_ Multi-modal Semantic Graph Prompt Learning Framework for Conversational Recommender Systems.pdf


In [2]:
# marker-pdf로 PDF → Markdown 변환
from marker.converters.pdf import PdfConverter
from marker.models import create_model_dict

converter = PdfConverter(artifact_dict=create_model_dict())
rendered = converter(pdf_path)

# rendered 객체 구조 확인
print(f"type: {type(rendered).__name__}")
print(f"속성: {[attr for attr in dir(rendered) if not attr.startswith('_')]}")

/Users/choseongyun/Documents/GitHub-SeongYun/ai-agent-master-class/wise-graduate-school-life/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
2026-04-05 01:17:42,775 [WARNING] surya: `TableRecEncoderDecoderModel` is not compatible with mps backend. Defaulting to cpu instead
Recognizing Text: 100%|██████████| 12/12 [00:05<00:00,  2.16it/s]


type: MarkdownOutput
속성: ['construct', 'copy', 'dict', 'from_orm', 'images', 'json', 'markdown', 'metadata', 'model_computed_fields', 'model_config', 'model_construct', 'model_copy', 'model_dump', 'model_dump_json', 'model_extra', 'model_fields', 'model_fields_set', 'model_json_schema', 'model_parametrized_name', 'model_post_init', 'model_rebuild', 'model_validate', 'model_validate_json', 'model_validate_strings', 'parse_file', 'parse_obj', 'parse_raw', 'schema', 'schema_json', 'update_forward_refs', 'validate']


In [ ]:
# 1) 메타데이터 확인
title = (
    rendered.metadata.get("title", Path(pdf_path).stem)
    if rendered.metadata
    else Path(pdf_path).stem
)
print(f"제목: {title}")
print(f"Markdown 길이: {len(rendered.markdown)} chars")
print(f"추출된 이미지 수: {len(rendered.images)}")
print(f"\n메타데이터 키: {list(rendered.metadata.keys()) if rendered.metadata else 'None'}")

In [ ]:
# 2) Markdown 미리보기 (앞 3000자)
print(rendered.markdown[:3000])

In [ ]:
# 3) 추출된 이미지 확인
for name, img in rendered.images.items():
    print(f"📷 {name} — size: {img.size}, mode: {img.mode}")
    display(img)

In [ ]:
# 4) Markdown → 섹션 분리
import re

def split_sections(markdown: str) -> list[dict]:
    """Markdown을 heading(#{1,3}) 기준으로 섹션 분리."""
    pattern = r"^(#{1,3})\s+(.+)$"
    lines = markdown.split("\n")

    sections = []
    current_heading = "Untitled"
    current_lines: list[str] = []

    for line in lines:
        match = re.match(pattern, line)
        if match:
            if current_lines:
                content = "\n".join(current_lines).strip()
                if content:
                    sections.append({"heading": current_heading, "content": content})
            current_heading = match.group(2).strip()
            current_lines = []
        else:
            current_lines.append(line)

    if current_lines:
        content = "\n".join(current_lines).strip()
        if content:
            sections.append({"heading": current_heading, "content": content})

    for i, s in enumerate(sections):
        s["section_index"] = i

    return sections

sections = split_sections(rendered.markdown)

print(f"총 {len(sections)}개 섹션\n")
for s in sections:
    preview = s["content"][:100].replace("\n", " ")
    print(f"[{s['section_index']}] {s['heading']}")
    print(f"    {preview}...")
    print()